# Prototyp vs. MQA: Indikator-Level-Vergleich

Reproduzierbarer Vergleich des Prototyps gegen den MQA-Baseline-Scorer auf identischem Input.
Logik in `src/evaluation/indicator_join.py`, Mapping in `src/evaluation/indicator_map.py`,
Konzept in `docs/evaluation_vs_mqa.md`.

**Drei Evidenzstücke:**
- **[A] Konvergenz (H1)** — Übereinstimmungsrate auf den geteilten strukturellen Checks
- **[B] Überschätzung (H2)** — `MQA PASS & Prototyp FAIL/PARTIAL` je Indikator
- **[C] Blindheit (H2)** — Prototyp-Score-Verteilung wo MQA keine Metrik hat

> **Vor dem Lauf:** `OPENROUTER_API_KEY` in `.env` für die expressiveness-Indikatoren (C).
> Ohne Key: `USE_LLM = False` setzen — die C-expr-Tabelle bleibt dann leer.

In [ ]:
import sys
from pathlib import Path
from datetime import datetime as _dt

REPO = Path.cwd()
while not (REPO / 'src').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))

from dotenv import load_dotenv
load_dotenv(REPO / '.env')

import pandas as pd
from IPython.display import display

# ===================== Parameter =====================
SAMPLE_DIR   = REPO / 'data' / 'sample_2026-06-25_09-57'  # <- Stichproben-Verzeichnis
USE_LLM      = True    # False: expr_*-Indikatoren = not_applicable (kein Token-Verbrauch)
USE_PROBE    = True     # False: HTTP-Checks deaktiviert (offline/schnell)
RECOMPUTE    = True     # True: Caches ignorieren und neu scoren

# Bereits gelaufenen Prototyp-Run wiederverwenden statt neu zu scoren:
# Pfad zu einem run_outputs/<run>-Verzeichnis (enthaelt <dataset>/result.json).
# None = Prototyp frisch scoren. Wenn gesetzt, wird nur die Prototyp-Seite
# geladen (PROTO_CONFIG/LLM dann fuers Modell irrelevant); MQA laeuft weiter auf
# SAMPLE_DIR, das dieselben .rdf-Dateien enthalten muss wie der Run.
# MODEL_RUN_DIR = None
MODEL_RUN_DIR = REPO / 'run_outputs' / 'run_2026-07-12_11-30-12_state-evaluation-final_EVAL'

# Prototyp-Konfiguration: Pfad zu einer State-YAML (conf/state/*.yaml) oder None für Defaults
# Beispiele:
#   PROTO_CONFIG = REPO / 'conf/state/state_evaluation_weighted.yaml'
#   PROTO_CONFIG = REPO / 'conf/state/state_presentation_tuned.yaml'
PROTO_CONFIG = REPO / 'conf/state/state_evaluation_weighted.yaml'     # None = Standardkonfiguration (keine Gewichtung, alle Dimensionen)
# =====================================================

_ts = _dt.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = REPO / 'comparison_output' / f'comparison_{SAMPLE_DIR.name}_{_ts}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('REPO      :', REPO)
print('SAMPLE    :', SAMPLE_DIR, '|', len(list(SAMPLE_DIR.glob('*.rdf'))), 'RDF-Dateien')
print('OUTPUT    :', OUTPUT_DIR)
print('LLM       :', 'an' if USE_LLM else 'aus (expr_* = N/A)')
print('HTTP-Probe:', 'an' if USE_PROBE else 'aus')
print('Config    :', PROTO_CONFIG.name if PROTO_CONFIG else '(Standard)')
print('Model-Run :', MODEL_RUN_DIR if MODEL_RUN_DIR else '(frisch scoren)')

In [ ]:
import yaml
from scoring.score_policy import ScorePolicy


def _service_kwargs_from_config(config_path):
    """Liest eine State-YAML und gibt die passenden QualityMetricsService-Kwargs zurück.

    Ausgewertete Felder: quality.{max_workers, dimension_weights, indicator_weights,
    dimension_whitelist, indicator_blacklist, indicator_whitelist, scoring} und language.
    Nicht gesetzte / leere Felder werden nicht übergeben → Service verwendet seine Defaults.
    """
    if config_path is None:
        print("PROTO_CONFIG : (Standard – keine YAML angegeben)")
        return {}

    p = Path(config_path)
    if not p.exists():
        raise FileNotFoundError(f"Config-Datei nicht gefunden: {p}")

    with open(p) as f:
        state = yaml.safe_load(f)

    q = state.get("quality", {})
    scoring_cfg = q.get("scoring")
    score_policy = ScorePolicy.from_config(scoring_cfg)

    kwargs = {}

    if q.get("max_workers") is not None:
        kwargs["max_workers"] = int(q["max_workers"])
    if q.get("dimension_weights"):
        kwargs["dimension_weights"] = dict(q["dimension_weights"])
    if q.get("indicator_weights"):
        kwargs["indicator_weights"] = dict(q["indicator_weights"])
    if q.get("dimension_whitelist"):
        kwargs["dimension_whitelist"] = list(q["dimension_whitelist"])
    if q.get("indicator_blacklist"):
        kwargs["indicator_blacklist"] = list(q["indicator_blacklist"])
    if q.get("indicator_whitelist"):
        kwargs["indicator_whitelist"] = list(q["indicator_whitelist"])
    if state.get("language"):
        kwargs["language"] = state["language"]
    if score_policy is not None:
        kwargs["score_policy"] = score_policy

    print(f"PROTO_CONFIG : {p.name}")
    print(f"  dimension_whitelist : {kwargs.get('dimension_whitelist', '(alle)')}")
    print(f"  dimension_weights   : {kwargs.get('dimension_weights', '(Standard)')}")
    print(f"  indicator_weights   : {'gesetzt (' + str(len(kwargs['indicator_weights'])) + ' Einträge)' if 'indicator_weights' in kwargs else '(Standard)'}")
    print(f"  score_policy        : {score_policy if score_policy else '(Standard)'}")
    return kwargs


SERVICE_KWARGS = _service_kwargs_from_config(PROTO_CONFIG)

## 1. Beide Modelle scoren (mit Cache)

Erster Lauf: beide Scorer laufen parallel auf jeder RDF-Datei.
Ergebnis wird in `model_indicators.csv` + `mqa_metrics.csv` gecacht — erneutes Ausführen
lädt den Cache (kein erneuter Token-Verbrauch), außer `RECOMPUTE = True`.

In [ ]:
from evaluation import ground_truth as gt
from evaluation import indicator_join as ij

# Kein LLM noetig, wenn die Prototyp-Seite aus einem fertigen Run geladen wird.
llm = gt.make_llm() if (USE_LLM and MODEL_RUN_DIR is None) else None

res = ij.build_all(
    SAMPLE_DIR,
    llm=llm,
    probe=USE_PROBE,
    recompute=RECOMPUTE,
    output_dir=OUTPUT_DIR,
    model_run_dir=MODEL_RUN_DIR,   # None = frisch scoren; sonst fertigen Run laden
    **SERVICE_KWARGS,
)
join = res['join']
print(f"join: {len(join)} Zeilen  ({join['file'].nunique()} Datensaetze x {join['indicator'].nunique()} Indikatoren)")
print(f"A/B/C-Verteilung: {res['class_counts']}")
if res['unmapped_indicators']:
    print('UNMAPPED:', res['unmapped_indicators'], '<- bitte in indicator_map.py eintragen')


## 2. A/B/C-Mapping-Übersicht

Alle Indikatoren, ihr MQA-Gegenpart und die Klassifikation. `review=True` markiert
Grenzfälle, die gegen die MQA-Spec belegt werden müssen.

In [ ]:
from evaluation.indicator_map import mapping_df, mqa_only_metrics

mdf = mapping_df()[['indicator','dimension','abc','mqa_str','review','rationale']]
mdf.columns = ['Indikator','Dimension','A/B/C','MQA-Metriken','Review?','Begründung']
display(mdf.sort_values(['A/B/C','Dimension']).reset_index(drop=True))

print('\nMQA prüft, Prototyp nicht (Balance §4.3):')
for m in mqa_only_metrics():
    print(f'  {m}')

## 3. [A] Konvergenz (H1)

Auf den **äquivalenten** Checks (A): stimmen MQA und Prototyp überein?
Erwartung: hohe Agreement-Rate → der Prototyp bildet die etablierte MQA-Logik korrekt ab.

In [ ]:
a = res['class_a']
if a['n'] == 0:
    print('Keine A-Paare vorhanden.')
else:
    print(f"[A] Konvergenz: {a['agreement']:.1%}  (n={a['n']} file×indicator-Paare)")
    print()
    contingency = pd.DataFrame({
        '': ['MQA PASS', 'MQA FAIL'],
        'Prototyp PASS': [a['both_pass'],       a['model_pass_mqa_fail']],
        'Prototyp FAIL': [a['mqa_pass_model_fail'], a['both_fail']],
    }).set_index('')
    display(contingency)

    # Per-A-Indikator aufschlüsseln
    a_detail = (
        join[join['abc'] == 'A']
        .assign(
            agree=lambda d: d['model_pass'] == d['mqa_pass'],
            mpf=lambda d: (d['mqa_pass'] == 1) & (d['model_pass'] == 0),
        )
        .groupby('indicator')
        .agg(n=('agree', 'count'), agreement=('agree', 'mean'), mqa_pass_model_fail=('mpf', 'sum'))
    )
    display(a_detail.round(2))

## 4. [B] Überschätzung durch MQA (H2)

Indikatoren, bei denen MQA = **PASS** (Presence genügt), der Prototyp aber **FAIL/PARTIAL**
(er prüft Gültigkeit/Tiefe/Kongruenz). Die obere linke Zelle einer Kontingenztabelle je
Indikator ist der direkte Beleg, dass MQA Qualität *annimmt*, die nicht da ist.

In [ ]:
b = res['class_b']
if b.empty:
    print('Keine B-Fälle.')
else:
    b_display = b.copy()
    b_display['überschätzung'] = b_display['overestimation_rate'].map('{:.0%}'.format)
    b_display['review?'] = b_display['review'].map({True: '⚠', False: ''})
    display(b_display[['indicator','n','mqa_pass_model_fail','überschätzung',
                        'both_pass','both_fail','model_pass_mqa_fail','review?']])

In [ ]:
# Beispieldatensätze: MQA PASS & Prototyp FAIL für den stärksten B-Indikator
if not b.empty:
    top_ind = b.iloc[0]['indicator']
    print(f'Disagreement-Fälle für {top_ind} (MQA=PASS, Prototyp=FAIL):')
    cases = join[
        (join['indicator'] == top_ind) &
        (join['mqa_pass'] == 1) &
        (join['model_pass'] == 0)
    ][['file','model_status','model_score']]
    display(cases.reset_index(drop=True))

## 5. [C] Strukturelle Blindheit der MQA (H2)

Für C-Indikatoren hat MQA **keine Metrik** — ihr Score ist konstruktiv flach (keine Reaktion).
Die Spalte `model_score_std` zeigt, ob der Prototyp hier tatsächlich variiert (= er erkennt
Unterschiede, die MQA per Design nicht sehen kann).

In [ ]:
c = res['class_c']
if c.empty:
    print('Keine C-Scores — entweder USE_LLM=False oder alle C-Indikatoren = not_applicable.')
else:
    c_display = c[['indicator','n','model_score_mean','model_score_std',
                   'model_score_min','model_score_max','n_below_0_5']].copy()
    c_display.columns = ['Indikator','n','Mittel','Std','Min','Max','<0.5']
    display(c_display.round(3))

## 6. Grafiken

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from IPython.display import Image, display as disp
from pathlib import Path

FIG_DIR = OUTPUT_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)
saved = []

def save(fig, name):
    p = FIG_DIR / f'{name}.png'
    fig.savefig(p, dpi=150, bbox_inches='tight')
    plt.close(fig)
    saved.append(p)
    return p


def _spearman(x, y):
    """Spearman ρ via rank-Pearson (keine scipy-Abhängigkeit)."""
    rx = np.argsort(np.argsort(x)).astype(float)
    ry = np.argsort(np.argsort(y)).astype(float)
    return float(np.corrcoef(rx, ry)[0, 1])


# ── 1. [B] Überschätzungsrate je B-Indikator (horizontale Balken) ────────────
if not b.empty:
    fig, ax = plt.subplots(figsize=(8, max(3, len(b) * 0.55)))
    colors = ['#d62728' if r > 0.25 else '#ff7f0e' if r > 0 else '#aec7e8'
              for r in b['overestimation_rate']]
    ax.barh(b['indicator'], b['overestimation_rate'], color=colors, alpha=0.85)
    ax.set_xlabel('Anteil MQA-PASS & Prototyp-FAIL (Überschätzungsrate)')
    ax.set_xlim(0, 1)
    ax.axvline(0, color='k', lw=0.8)
    for i, (_, r) in enumerate(b.iterrows()):
        ax.text(r['overestimation_rate'] + 0.01, i,
                f"{int(r['mqa_pass_model_fail'])}/{int(r['n'])}", va='center', fontsize=9)
    ax.set_title('[B] MQA-Überschätzung je Indikator')
    ax.invert_yaxis()
    fig.tight_layout()
    save(fig, 'b_overestimation')
    disp(Image(str(FIG_DIR / 'b_overestimation.png')))


# ── 2. [A] Konvergenz als 2×2-Konfusionsmatrix ───────────────────────────────
if a['n'] > 0:
    mat = np.array([
        [a['both_pass'],            a['model_pass_mqa_fail']],
        [a['mqa_pass_model_fail'],  a['both_fail']],
    ], dtype=float)
    total = mat.sum()
    agree_mask = np.array([[True, False], [False, True]])
    norm = mat / total

    fig, ax = plt.subplots(figsize=(5, 4.5))
    for i in range(2):
        for j in range(2):
            base = '#2ca02c' if agree_mask[i, j] else '#d62728'
            alpha = 0.15 + 0.75 * norm[i, j]
            rgba = (*mcolors.to_rgb(base), alpha)
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, color=rgba, zorder=1))
            count = int(mat[i, j])
            pct   = 100 * norm[i, j]
            ax.text(j, i - 0.08, str(count),
                    ha='center', va='center', fontsize=18, fontweight='bold', zorder=2)
            ax.text(j, i + 0.22, f'{pct:.1f} %',
                    ha='center', va='center', fontsize=10, color='#333333', zorder=2)

    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['MQA PASS', 'MQA FAIL'], fontsize=11)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Prototyp\nPASS', 'Prototyp\nFAIL'], fontsize=11)
    ax.xaxis.set_label_position('top')
    ax.xaxis.tick_top()
    ax.set_title(
        f'[A] Konvergenz auf geteilten Checks\n'
        f'Übereinstimmung {a["agreement"]:.1%}  (n = {a["n"]} file×indicator)',
        fontsize=11, pad=14,
    )
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(length=0)
    fig.tight_layout()
    save(fig, 'a_agreement_matrix')
    disp(Image(str(FIG_DIR / 'a_agreement_matrix.png')))


# ── 3. [C] Boxplot der Prototyp-Scores je C-Indikator ────────────────────────
if not c.empty:
    c_inds = c['indicator'].tolist()
    c_data = [join.loc[(join['indicator'] == ind) & join['model_score'].notna(),
                       'model_score'].astype(float).tolist() for ind in c_inds]
    c_data_filt = [d for d in c_data if d]
    c_inds_filt = [ind for ind, d in zip(c_inds, c_data) if d]
    c_labels = [
        i.replace('expr_', 'expr.\n').replace('acc_', 'acc.\n')
         .replace('find_', 'find.\n').replace('reuse_', 'reuse.\n')
        for i in c_inds_filt
    ]
    if c_data_filt:
        fig, ax = plt.subplots(figsize=(max(7, len(c_data_filt) * 1.05), 5))
        ax.boxplot(c_data_filt, labels=c_labels, patch_artist=True,
                   boxprops=dict(facecolor='#aec7e8', alpha=0.8),
                   medianprops=dict(color='#1f77b4', lw=2))
        ax.axhline(0.5, color='#d62728', lw=1, ls='--', label='Score 0.5')
        ax.set_ylabel('Prototyp-Score (0–1)')
        ax.set_ylim(-0.05, 1.05)
        ax.set_title('[C] Prototyp-Score-Verteilung — MQA hat keine Metrik')
        ax.legend(fontsize=8)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
        fig.tight_layout()
        save(fig, 'c_blindness_boxplot')
        disp(Image(str(FIG_DIR / 'c_blindness_boxplot.png')))


# ── 4. Gesamt-Scatter: Prototyp vs. MQA mit Korrelation & Bias ───────────────
mqa_raw = pd.read_csv(OUTPUT_DIR / 'mqa_metrics.csv')
mqa_per_file = (
    mqa_raw.groupby('file')
    .apply(lambda g: pd.Series({
        'mqa_score_norm': g['mqa_points'].sum() / g['mqa_max'].sum() if g['mqa_max'].sum() > 0 else float('nan')
    }))
    .reset_index()
)

model_raw = pd.read_csv(OUTPUT_DIR / 'model_indicators.csv')
model_per_file = (
    model_raw[model_raw['model_pass'].notna()]
    .groupby('file')['model_pass']
    .mean()
    .reset_index()
    .rename(columns={'model_pass': 'model_score_norm'})
)

scatter_df = mqa_per_file.merge(model_per_file, on='file')

if not scatter_df.empty:
    x = scatter_df['mqa_score_norm'].values
    y = scatter_df['model_score_norm'].values
    valid = ~(np.isnan(x) | np.isnan(y))
    xv, yv = x[valid], y[valid]

    pearson_r  = float(np.corrcoef(xv, yv)[0, 1])
    spearman_r = _spearman(xv, yv)
    mean_diff  = float((yv - xv).mean())       # Prototyp − MQA (negativ = Prototyp strenger)
    median_diff = float(np.median(yv - xv))

    # Regressionsgerade
    m, b_coef = np.polyfit(xv, yv, 1)
    x_line = np.linspace(xv.min(), xv.max(), 100)

    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.scatter(xv, yv, s=55, alpha=0.75, color='#1f77b4', edgecolors='white', lw=0.4, label='Datensätze')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4, label='y = x (Identität)')
    ax.plot(x_line, m * x_line + b_coef, color='#d62728', lw=1.5, alpha=0.8,
            label=f'Regression (m = {m:.2f})')

    # Stats-Box oben links
    stats_text = (
        f'Pearson  r = {pearson_r:+.3f}\n'
        f'Spearman ρ = {spearman_r:+.3f}\n'
        f'Ø Δ (Prot − MQA) = {mean_diff:+.3f}\n'
        f'Median Δ          = {median_diff:+.3f}'
    )
    ax.text(0.03, 0.97, stats_text, transform=ax.transAxes,
            va='top', ha='left', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='#cccccc', alpha=0.9),
            fontfamily='monospace')

    ax.set_xlabel('MQA normalisiert (0–1)')
    ax.set_ylabel('Prototyp Pass-Rate (0–1)')
    ax.set_title(f'Gesamt-Scores: Prototyp vs. MQA  (n = {len(xv)})')
    ax.legend(fontsize=8, loc='lower right')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    fig.tight_layout()
    save(fig, 'overall_scatter')
    disp(Image(str(FIG_DIR / 'overall_scatter.png')))

    # Kennzahlen auch als Text ausgeben
    print(f'\nKorrelation Prototyp ↔ MQA (n={len(xv)}):')
    print(f'  Pearson  r  = {pearson_r:+.3f}')
    print(f'  Spearman ρ  = {spearman_r:+.3f}')
    print(f'  Ø Δ (Prot − MQA) = {mean_diff:+.3f}  (negativ = Prototyp strenger)')
    print(f'  Median Δ          = {median_diff:+.3f}')

print('\nGrafiken gespeichert:')
for p in saved:
    print(' ', p)

## 7. Disagreement-Analyse

Datensätze nach größter Score-Differenz ranken — die Top-Fälle zeigen **warum** die Modelle
auseinanderlaufen und liefern Material für die qualitative Diskussion (vgl. §6 `docs/evaluation_vs_mqa.md`).

In [ ]:
if not scatter_df.empty:
    scatter_df['diff'] = scatter_df['model_score_norm'] - scatter_df['mqa_score_norm']
    scatter_df['abs_diff'] = scatter_df['diff'].abs()
    print('=== Top Disagreements (|Prototyp - MQA| größte Abweichung) ===')
    display(
        scatter_df.sort_values('abs_diff', ascending=False)
        [['file','mqa_score_norm','model_score_norm','diff']]
        .rename(columns={'mqa_score_norm':'MQA','model_score_norm':'Prototyp','diff':'Δ (Prot-MQA)'})
        .round(3)
        .head(15)
        .reset_index(drop=True)
    )

    # Für jeden Top-Disagrement: welche B-Indikatoren sind Schuld?
    top_file = scatter_df.sort_values('abs_diff', ascending=False).iloc[0]['file']
    print(f'\nB-Indikatoren für "{top_file}":')
    display(
        join[
            (join['file'] == top_file) & (join['abc'] == 'B')
        ][['indicator','model_status','model_score','mqa_metrics','mqa_pass']]
        .reset_index(drop=True)
    )

## 8. Zusammenfassung

Kompakte Übersicht aller Headline-Zahlen für die Thesis.

In [ ]:
print('=== Zusammenfassung Prototyp vs. MQA ===')
print(f'Stichprobe : {join["file"].nunique()} Datensätze, {join["indicator"].nunique()} Indikatoren')
print(f'A/B/C      : {res["class_counts"]}')
print()
if a['n']:
    print(f'[A] Konvergenz   : {a["agreement"]:.1%}  (MQA-PASS/Modell-FAIL: {a["mqa_pass_model_fail"]}, '
          f'Modell-PASS/MQA-FAIL: {a["model_pass_mqa_fail"]})')

if not b.empty:
    total_overest = int(b['mqa_pass_model_fail'].sum())
    total_b_cases = int(b['n'].sum())
    print(f'[B] Überschätzung: {total_overest}/{total_b_cases} B-Fälle ({total_overest/total_b_cases:.1%}) '
          f'MQA-PASS & Prototyp-FAIL')
    top = b.iloc[0]
    print(f'   Stärkster B-Indikator: {top["indicator"]} ({int(top["mqa_pass_model_fail"])}/{int(top["n"])}, '
          f'{top["overestimation_rate"]:.0%})')

if not c.empty:
    n_active_c = int(c['n_below_0_5'].sum())
    n_total_c = int(c['n'].sum())
    print(f'[C] Blindheit    : {c["indicator"].nunique()} Indikatoren ohne MQA-Pendant, '
          f'{n_active_c}/{n_total_c} Fälle Score<0.5')

print(f'\nMQA-only-Metriken (Prototyp nicht): {res["mqa_only_metrics"]}')
if res['unmapped_indicators']:
    print(f'Unmapped: {res["unmapped_indicators"]}')